In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
block_size = 8
batch_size = 4
vocab_size = 80
max_iters = 10000 
learning_rate = 3e-4

In [3]:
with open("./data/wizard_of_oz.txt",'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
print(len(text))

232316


In [5]:
chars = sorted(list(set(text)))
print(len(chars))
print(''.join(chars))

81

 !"&'()*,-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz﻿


In [6]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: [itos[i] for i in l] 

In [7]:
print(encode(text[:10]))
print(''.join(decode(encode(text[:10]))))

[80, 1, 1, 28, 39, 42, 39, 44, 32, 49]
﻿  DOROTHY


In [8]:
data = torch.tensor(encode(text),dtype=torch.long)
print(data[:100])

tensor([80,  1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,
         1, 47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26,
        49,  0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,
         0,  0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1,
        36, 25, 38, 28,  1, 39, 30,  1, 39, 50])


In [9]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for i in range(block_size):
    context = x[:i+1]
    target = y[i]
    print(f"For context {context} the target is {target}")

For context tensor([80]) the target is 1
For context tensor([80,  1]) the target is 1
For context tensor([80,  1,  1]) the target is 28
For context tensor([80,  1,  1, 28]) the target is 39
For context tensor([80,  1,  1, 28, 39]) the target is 42
For context tensor([80,  1,  1, 28, 39, 42]) the target is 39
For context tensor([80,  1,  1, 28, 39, 42, 39]) the target is 44
For context tensor([80,  1,  1, 28, 39, 42, 39, 44]) the target is 32


In [17]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(0,len(data)-block_size,size=(batch_size,)) # block_size=8 , batch_size=4
    x = torch.stack([data[i:block_size+i] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

In [12]:
x, y = get_batch('train')
print(f"inputs\n{x}")
print(f"targets\n{y}")

tensor([124644, 177625,   2497,  50261])
inputs
tensor([[ 1, 73, 61, 58, 62, 71,  1, 66],
        [61, 62, 72,  1, 69, 65, 54, 56],
        [76, 62, 65, 65,  0, 54, 65, 72],
        [57,  1, 73, 68,  1, 71, 58, 54]])
targets
tensor([[73, 61, 58, 62, 71,  1, 66, 58],
        [62, 72,  1, 69, 65, 54, 56, 58],
        [62, 65, 65,  0, 54, 65, 72, 68],
        [ 1, 73, 68,  1, 71, 58, 54, 65]])


#### @ BiGramLanguageModel

In [13]:
class bigramlanguagemodel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)
        
    def forward_pass(self,index,targets=None):
        logits = self.token_embedding_table(index)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)

        return logits,loss
    
    def generate(self,index,max_new_tokens):  
        for _ in range(max_new_tokens):
            logits, _ = self.forward_pass(index) 
            logits = logits[:,-1,:] # becomes (B,C)
            probs = F.softmax(logits,dim=-1)
            next_index = torch.multinomial(probs,num_samples=1) # (B,1)
            index = torch.cat((index,next_index),dim=1) # (B,T+1) 
        return index


In [ ]:
# This code generates garbage tokens(characters) right now as we still haven't trained the model yet.
model = bigramlanguagemodel(vocab_size=vocab_size)
m = model.to(device)
 
context = torch.zeros(size=(1,1),dtype=torch.long,device=device)
generated_chars = decode(m.generate(context,500)[0].tolist())
generated_chars = ''.join(generated_chars)
print(generated_chars)


5ww;fU:;yO[luML'a6HVQXUS'Rln'n63DjxT6Qi!P0BwH-!05?m!2[i!cxFjorJ8jubnq6:"*
52k rG.DeQV2DeJ9Z30'sfxBTvl,YR, 'nBb(xBirLAyusRXNFtp'rJ1D(?RYYRl3:O,OglHBq)9bU2wv*
ETpdXky6:k]5kIi-dUS9_jzp:dpdoo6Y)WZ3f "&Gr)w SA9XlZ;Y:-5b;QT6I(jw]z7"[SQwy)-FXKCC;xUDr ep8rEfAh-&HHIY?U5jSSpdgJ1_j6[.kyHI5N2t' x2o,1!1'Z6dPkyi?V*8?8u-:JSeeJw9
]Yl_L Enb;_lHI?8YV6I?dmE;()dT.KtL!5[gY6*vKG'!".D(KCG&:rMCwvd8dT''&Ed5Oo'XO)CET(UXP
r)V3C,]iGz6SL 0oAPuh)9DZL[-NF8(kf5R
,H-*Jx_jwXKVnRb
P:G&'3ja6u9T"KmHV4:hSG9-jDZ3c?Rb4C6N4H4Kg9o?N-0ev


#### @ Loss Optimizer

In [22]:
optimizer = torch.optim.AdamW(model.parameters(),learning_rate)

for iters in range(max_iters):
    xb,yb = get_batch('train')

    logits,loss = model.forward_pass(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


2.2685370445251465


In [24]:
# As you can see once we optimized the loss function to some extent the characters start making some sense compared to previous garbage values 
context = torch.zeros(size=(1,1),dtype=torch.long,device=device)
generated_chars = decode(m.generate(context,500)[0].tolist())
generated_chars = ''.join(generated_chars)
print(generated_chars)


b, ithefa

FQTuram, hanl heil. BOde t."bakie p lly _wase
ARDoulay med, th pphe MQ*ugnd I t!H. "
he om rmad, wha
ge THL. y, tun e rheabut?dlowend g thtthexWhetherthangal I8AZU. s co Cnond tllthe f lair?xenomos d ther inrdad oupt tcked we,"coumurlanvain MZ(us DI"YKwharsimncapookimathe be b.
d, H9NHI'toresco Ipwa the ldime?R?
"

warpouse cllcain t intind BvETKCnd oo thede ha thepther f
" Hildm'use tofuricyrontig-&' Lz[xw.
e s?. m w mind tidd it ownggioneaizLpowed was,"OTF;ut hesh1Ahe por IBe w m
SS
